# Structured Outputs with Claude on Amazon Bedrock

Structured outputs make Claude return JSON that conforms to a JSON Schema you supply. The
schema is compiled into a grammar that constrains decoding, so fields that violate the
schema are not merely discouraged — the tokens that would produce them are unreachable.
The documented exceptions (refusals, truncation at `max_tokens`, and enum casing) are
covered in Part 4.

This removes the usual post-processing tax: no stripping ```` ```json ```` fences, no
retry-on-parse-failure loop, no defensive `.get()` chains for fields the model forgot.

## What this notebook covers

Claude on Bedrock exposes structured outputs through two endpoints and several APIs. The
request shape differs on each, which is the main thing this notebook pins down.

| Endpoint | API | Where the schema goes |
|---|---|---|
| `bedrock-runtime` | `Converse` | `outputConfig.textFormat.structure.jsonSchema.schema` |
| `bedrock-runtime` | `ConverseStream` | same as `Converse` |
| `bedrock-runtime` | `InvokeModel` | `output_config.format.schema` |
| `bedrock-runtime` | `InvokeModelWithResponseStream` | same as `InvokeModel` |
| `bedrock-mantle` | Messages (Anthropic-native) | `output_config.format.schema` |

Plus **strict tool use** on each, which applies the same grammar guarantee to tool inputs
rather than to the assistant's text.

We use a single model throughout — Claude Sonnet 5 — so the only thing changing between
examples is the API surface.

## Setup

Structured outputs on `bedrock-runtime` need a recent `boto3`/`botocore`; the
`bedrock-mantle` endpoint is called with the Anthropic SDK.

In [ ]:
%pip install -qU --disable-pip-version-check boto3 botocore "anthropic>=1.13.0"

In [ ]:
import json

import boto3
from botocore.config import Config

REGION = "us-east-1"

# bedrock-runtime requires an inference profile prefix ("global.", or a geo prefix such as
# "us."/"eu."). A bare model ID fails with "on-demand throughput isn't supported".
RUNTIME_MODEL = "global.anthropic.claude-sonnet-5"

# bedrock-mantle uses short, unprefixed, unversioned IDs. Adding "global."/"us." or a
# version suffix returns 404.
MANTLE_MODEL = "anthropic.claude-sonnet-5"

bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name=REGION,
    config=Config(read_timeout=300),
)

## The schema

One schema, reused by every example below. Two rules are worth knowing up front:

- **`additionalProperties: false` is mandatory** on every object. Omitting it — or setting
  it to `true` — is a validation error, not a silently relaxed default.
- **Optional properties are allowed, but they cost.** A property left out of `required`
  may be omitted from the output, and required properties are always emitted before
  optional ones. Each optional property counts toward a limit of 24 per request and
  enlarges the compiled grammar. If you want a key that is always present but may be
  empty, keep it in `required` and give it a nullable type, e.g.
  `{"type": ["string", "null"]}` (union types have their own limit of 16 per request).

`description` on a property is not decoration: it is the only instruction the model gets
about what belongs in that field, so it does real work.

In [ ]:
LEAD_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Customer name"},
        "email": {"type": "string", "description": "Customer email address"},
        "plan_interest": {"type": "string", "description": "Product plan of interest"},
        "demo_requested": {
            "type": "boolean",
            "description": "Whether a demo was requested",
        },
    },
    "required": ["name", "email", "plan_interest", "demo_requested"],
    "additionalProperties": False,
}

SYSTEM = "Extract CRM lead fields from the message."

PROMPT = (
    "Paulo Santos (paulo@example.com) is interested in our Enterprise plan "
    "and wants to schedule a demo for next Tuesday at 2pm."
)

---

# Part 1 — `bedrock-runtime`: the Converse API

`Converse` is the cross-model Bedrock API. Structured outputs are configured with the
`outputConfig` parameter.

**The one thing that trips people up:** `jsonSchema.schema` is typed as a *string*, not a
document. You must pass `json.dumps(schema)`. Handing it a `dict` fails botocore's local
parameter validation before any request is sent:

```
Invalid type for parameter outputConfig.textFormat.structure.jsonSchema.schema
```

`textFormat.type` is also required (`"json_schema"` is its only value today), and
`textFormat.structure` is required alongside it.

In [ ]:
response = bedrock_runtime.converse(
    modelId=RUNTIME_MODEL,
    system=[{"text": SYSTEM}],
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    inferenceConfig={"maxTokens": 512},
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {
                # Note json.dumps: this field takes a STRING, not a dict.
                "jsonSchema": {"schema": json.dumps(LEAD_SCHEMA)}
            },
        }
    },
)

# Select the text block by key: content[0] is not guaranteed to be text (it may be a
# reasoning block), which is the same caveat as InvokeModel in Part 2.
text = next(b["text"] for b in response["output"]["message"]["content"] if "text" in b)
lead = json.loads(text)

print(json.dumps(lead, indent=2))
print("\nstopReason:", response["stopReason"])
print("usage:", response["usage"])

`Converse` also accepts optional `name` and `description` alongside `schema`, which are
useful for labelling the schema in logs and traces:

```python
"jsonSchema": {
    "name": "lead_extraction",
    "description": "Extract CRM lead fields",
    "schema": json.dumps(LEAD_SCHEMA),
}
```

Worth knowing: these two keys are **accepted on `Converse` but rejected on the
Anthropic-native surfaces** (`InvokeModel` and the Mantle Messages API), which return
`output_config.format.name: Extra inputs are not permitted`.

### A note on `temperature`

Newer Claude models reject `inferenceConfig.temperature` outright
(`` `temperature` is deprecated for this model ``). Set `maxTokens` and leave sampling
parameters alone.

## Converse with streaming

`ConverseStream` takes the identical `outputConfig`. The deltas arrive as ordinary text
chunks; concatenate them and parse once the stream closes. Partial JSON is not valid JSON,
so there is nothing to parse mid-stream unless you add an incremental parser.

In [ ]:
stream = bedrock_runtime.converse_stream(
    modelId=RUNTIME_MODEL,
    messages=[{"role": "user", "content": [{"text": PROMPT}]}],
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {"jsonSchema": {"schema": json.dumps(LEAD_SCHEMA)}},
        }
    },
)

buffer = ""
for event in stream["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"].get("text", "")
        buffer += chunk
        print(chunk, end="")

print("\n\nparsed:", json.loads(buffer))

## Converse with strict tool use

`outputConfig` constrains the assistant's *text*. To get the same guarantee on *tool
inputs*, set `strict: True` on the tool spec. Without it, `inputSchema` is a strong hint;
with it, the schema is compiled into a grammar exactly as above — so Claude cannot omit a
required argument or pass one the schema doesn't declare.

**The shapes are inconsistent here, so read carefully:** `toolSpec.inputSchema.json` takes
a **dict**, while `outputConfig`'s `jsonSchema.schema` takes a **JSON string**. Passing a
string to `inputSchema.json` fails with
`The format of the value at toolConfig.tools.0.toolSpec.inputSchema.json is invalid`.

In [ ]:
WEATHER_SCHEMA = {
    "type": "object",
    "properties": {
        "location": {"type": "string", "description": "City and state"},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
    },
    "required": ["location", "unit"],
    "additionalProperties": False,
}

response = bedrock_runtime.converse(
    modelId=RUNTIME_MODEL,
    messages=[
        {
            "role": "user",
            "content": [{"text": "What's the weather in San Francisco in celsius?"}],
        }
    ],
    toolConfig={
        "tools": [
            {
                "toolSpec": {
                    "name": "get_weather",
                    "description": "Get the current weather for a location",
                    "strict": True,
                    # Note: a dict here, NOT json.dumps(...).
                    "inputSchema": {"json": WEATHER_SCHEMA},
                }
            }
        ]
    },
)

for block in response["output"]["message"]["content"]:
    if "toolUse" in block:
        print(block["toolUse"]["name"], "->", block["toolUse"]["input"])

print("\nstopReason:", response["stopReason"])

---

# Part 2 — `bedrock-runtime`: the InvokeModel API

`InvokeModel` passes an Anthropic-native request body straight through, so it uses
Anthropic's own field names: `output_config.format` instead of `outputConfig.textFormat`.

Two differences from `Converse` worth internalising:

- `schema` here is a **dict** — the whole body is already JSON, so there is no nested
  string.
- `max_tokens` and `anthropic_version` are required body fields rather than
  `inferenceConfig` entries.

In [ ]:
response = bedrock_runtime.invoke_model(
    modelId=RUNTIME_MODEL,
    body=json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "system": SYSTEM,
            "messages": [{"role": "user", "content": PROMPT}],
            "output_config": {
                "format": {
                    "type": "json_schema",
                    # A dict here, unlike Converse.
                    "schema": LEAD_SCHEMA,
                }
            },
        }
    ),
)

body = json.loads(response["body"].read())

# content[0] is not guaranteed to be the text block (it may be a thinking block),
# so select by type.
text = next(b["text"] for b in body["content"] if b["type"] == "text")

print(json.dumps(json.loads(text), indent=2))
print("\nstop_reason:", body["stop_reason"])
print("usage:", body["usage"])

## InvokeModel with streaming

`InvokeModelWithResponseStream` emits Anthropic-native server-sent events. The text lives
in `content_block_delta` events.

In [ ]:
response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=RUNTIME_MODEL,
    body=json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "messages": [{"role": "user", "content": PROMPT}],
            "output_config": {
                "format": {"type": "json_schema", "schema": LEAD_SCHEMA}
            },
        }
    ),
)

buffer = ""
for event in response["body"]:
    data = json.loads(event["chunk"]["bytes"])
    if data.get("type") == "content_block_delta":
        chunk = data["delta"].get("text", "")
        buffer += chunk
        print(chunk, end="")

print("\n\nparsed:", json.loads(buffer))

## InvokeModel with strict tool use

Anthropic-native tools use `input_schema` (not `parameters`), take `strict` as a
sibling key, and have **no** `"type": "function"` wrapper — that shape belongs to
OpenAI-compatible APIs and is rejected here.

In [ ]:
response = bedrock_runtime.invoke_model(
    modelId=RUNTIME_MODEL,
    body=json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "tools": [
                {
                    "name": "get_weather",
                    "description": "Get the current weather for a location",
                    "strict": True,
                    "input_schema": WEATHER_SCHEMA,
                }
            ],
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in San Francisco in celsius?",
                }
            ],
        }
    ),
)

body = json.loads(response["body"].read())

for block in body["content"]:
    if block["type"] == "tool_use":
        print(block["name"], "->", block["input"])

print("\nstop_reason:", body["stop_reason"])

---

# Part 3 — the `bedrock-mantle` endpoint: Messages API

`bedrock-mantle` serves Anthropic's own Messages API, authenticated with SigV4 against
your AWS credentials. You get the Anthropic SDK's ergonomics — typed responses, a context
manager for streaming — while staying inside AWS IAM.

Use `AnthropicBedrockMantle`, which signs requests for you. Note the model ID convention
differs from `bedrock-runtime`: short, **unprefixed and unversioned**. `global.`/`us.`
prefixes and `-v1:0` suffixes both 404 here.

Model availability also differs between the two endpoints, so confirm your model is served
on `bedrock-mantle` in your Region rather than assuming parity with `bedrock-runtime`.

> **Check the docs before relying on this path in production.** At the time of writing,
> the [Amazon Bedrock structured outputs guide](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html)
> lists `output_config.format` on the Mantle Messages API as unsupported and points to
> `Converse` or `InvokeModel` instead. The calls below succeeded and returned
> schema-conforming output for Claude Sonnet 5 in `us-east-1` when this notebook was
> tested, but Parts 1 and 2 are the documented path.

In [ ]:
from anthropic import AnthropicBedrockMantle

mantle = AnthropicBedrockMantle(aws_region=REGION)

print("endpoint:", mantle.base_url)

In [ ]:
message = mantle.messages.create(
    model=MANTLE_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{"role": "user", "content": PROMPT}],
    output_config={"format": {"type": "json_schema", "schema": LEAD_SCHEMA}},
)

text = next(block.text for block in message.content if block.type == "text")

print(json.dumps(json.loads(text), indent=2))
print("\nstop_reason:", message.stop_reason)
print("usage:", message.usage)

## Mantle with streaming

`messages.stream()` gives you a `text_stream` helper, so there is no event-type
dispatching to write.

In [ ]:
chunks = []
with mantle.messages.stream(
    model=MANTLE_MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": PROMPT}],
    output_config={"format": {"type": "json_schema", "schema": LEAD_SCHEMA}},
) as stream:
    for text in stream.text_stream:
        chunks.append(text)
        print(text, end="")

print("\n\nparsed:", json.loads("".join(chunks)))

## Mantle with strict tool use

Identical tool shape to `InvokeModel`, since both speak the Anthropic-native dialect.

In [ ]:
message = mantle.messages.create(
    model=MANTLE_MODEL,
    max_tokens=512,
    tools=[
        {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "strict": True,
            "input_schema": WEATHER_SCHEMA,
        }
    ],
    messages=[
        {"role": "user", "content": "What's the weather in San Francisco in celsius?"}
    ],
)

for block in message.content:
    if block.type == "tool_use":
        print(block.name, "->", block.input)

print("\nstop_reason:", message.stop_reason)

---

# Part 4 — Handling responses in production

A structured output guarantees the *shape* of a successful completion. It does not
guarantee that you got one, and it has one documented gap. These cases need explicit
handling:

- **`max_tokens`** — the grammar forces well-formed JSON, but a truncated response is
  still truncated. You will get invalid JSON. Treat it as a retry with a higher budget,
  not a parse error.
- **`refusal`** — Claude declined. There is no JSON to parse. This matters most when the
  input is untrusted user content.
- **`malformed_model_output`** — a `Converse`-only stop reason. The API reference lists it
  without further description; treat it as a failed extraction, not as parseable output.
- **Enum casing** — structured outputs don't guarantee the capitalization of `enum` and
  `const` string values. Claude may return `"Topic Two"` for an enum value of
  `"Topic two"`, with a normal stop reason. Compare enum values case-insensitively, and
  avoid values that differ only in case.

The stop reason vocabularies are not identical across the two surfaces:

| Surface | Stop reasons |
|---|---|
| Anthropic-native (`InvokeModel`, Mantle) | `end_turn`, `max_tokens`, `stop_sequence`, `tool_use`, `pause_turn`, `refusal`, `model_context_window_exceeded` |
| `Converse` | `end_turn`, `max_tokens`, `stop_sequence`, `tool_use`, `guardrail_intervened`, `content_filtered`, `malformed_model_output`, `malformed_tool_use`, `model_context_window_exceeded` |

Note that `Converse` has no `refusal` and Anthropic-native has no
`malformed_model_output`, so a handler written against one surface will not be complete on
the other. How a Claude refusal surfaces through `Converse` is not documented, so guard the
`json.loads` there too rather than trusting `end_turn` alone.

In [ ]:
def parse_structured_output(body):
    """Parse an Anthropic-native structured response, or explain why we can't.

    Accepts the decoded InvokeModel body (a dict) or an SDK Message.
    Returns (data, error) where exactly one is None.
    """
    if isinstance(body, dict):
        stop_reason = body["stop_reason"]
        blocks = [(b["type"], b.get("text")) for b in body["content"]]
    else:  # anthropic SDK Message
        stop_reason = body.stop_reason
        blocks = [(b.type, getattr(b, "text", None)) for b in body.content]

    if stop_reason == "refusal":
        return None, "model declined to respond; route to manual review"
    if stop_reason == "max_tokens":
        return None, "truncated before the JSON closed; retry with a larger max_tokens"
    if stop_reason not in ("end_turn", "stop_sequence"):
        return None, f"unexpected stop_reason: {stop_reason}"

    text = next((t for kind, t in blocks if kind == "text"), None)
    if text is None:
        return None, f"no text block in response: {[k for k, _ in blocks]}"

    try:
        return json.loads(text), None
    except json.JSONDecodeError as exc:
        return None, f"text block is not valid JSON: {exc}"


# Exercise it against a real call.
response = bedrock_runtime.invoke_model(
    modelId=RUNTIME_MODEL,
    body=json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 512,
            "messages": [{"role": "user", "content": PROMPT}],
            "output_config": {
                "format": {"type": "json_schema", "schema": LEAD_SCHEMA}
            },
        }
    ),
)

data, error = parse_structured_output(json.loads(response["body"].read()))
print("data:", data)
print("error:", error)

---
# Part 5 — Schema features and limits

The schema is compiled into a grammar, and that compiler supports a deliberate subset of
JSON Schema. Most unsupported keywords are **rejected at request time** with a
`ValidationException` rather than ignored, so you find out immediately rather than in
production. The lists below combine the
[Anthropic](https://platform.claude.com/docs/en/build-with-claude/structured-outputs#json-schema-limitations)
and [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html)
docs with what `bedrock-runtime` returned for Claude Sonnet 5 when this notebook was
tested. Check the docs for the current list.

Supported, and enforced during decoding:
- `type` (including `"null"` and union lists such as `["string", "null"]`)
- `enum` (strings, numbers, booleans, or nulls only), `const`
- `anyOf`, `allOf` (`allOf` combined with `$ref` is not supported)
- `required`, `additionalProperties: false`
- `properties`, nested objects, `items`
- `$defs` / `definitions` with internal `$ref`
- `pattern`, limited to simple regexes (no backreferences, lookaround, or `\b`)
- string `format`: `date-time`, `time`, `date`, `duration`, `email`, `hostname`, `uri`,
  `ipv4`, `ipv6`, `uuid`
- `minItems` of `0` or `1`

Commonly reached for, but **rejected**:
- numeric bounds — `minimum`, `maximum`, `exclusiveMinimum`, `exclusiveMaximum`, `multipleOf`
- array constraints — `maxItems`, `uniqueItems`, `prefixItems`, `minItems` of 2 or more
- object constraints — `minProperties`, `maxProperties`, `patternProperties`, `propertyNames`
- composition — `oneOf`, `not`
- recursive schemas (a `$ref` that points back to itself, directly or indirectly)
- `additionalProperties: true`, or omitting it entirely
- external (URL) `$ref`

Accepted by `bedrock-runtime` but **not guaranteed** — validate these yourself if you
depend on them:
- `minLength`, `maxLength` — both docs list these as unsupported, yet requests that use
  them currently succeed. In testing, a `minLength: 120` field came back 119 characters
  long.
- `if`/`then`/`else` — undocumented; accepted in testing, but don't build on it.
- annotations: `default`, `examples`, `title`, `readOnly`, `deprecated`, `$schema`

Enforce what the grammar won't by validating client-side:
```python
# pip install jsonschema
import jsonschema
jsonschema.validate(instance=lead, schema=LEAD_SCHEMA)
```

The Anthropic SDK's `anthropic.transform_schema()` handles the other half: it removes
unsupported constraints from a schema (or Pydantic model) and moves them into field
descriptions, so you can send a schema the API accepts and still validate against the
original.

## Grammar compilation

Compiling a schema into a grammar has a one-time cost: the first request with a new schema
takes longer (the Bedrock docs say up to a few minutes). Compiled grammars are cached per
account for 24 hours, so later requests with an identical schema skip it. **Keep your
schema object stable** — building it dynamically per request defeats the cache.

Complexity is capped. These limits apply across all strict tools and JSON output schemas
in a single request:

| Limit | Value |
|---|---|
| Tools with `strict: true` | 20 |
| Optional parameters (not listed in `required`) | 24 |
| Parameters with union types (`anyOf`, or type arrays like `["string", "null"]`) | 16 |

A schema can stay within all three and still fail with `The compiled grammar is too
large`, because nesting, optional fields, and unions multiply each other. If you hit it,
reduce optional and union-typed parameters, flatten nested objects, or mark fewer tools as
strict, rather than retrying.

## On the OpenAI-compatible routes

`bedrock-mantle` also exposes OpenAI-compatible Responses and Chat Completions APIs, and
structured outputs work on them for models those routes serve:

| API | Schema location |
|---|---|
| Responses | `text.format` — `{"type": "json_schema", "name": ..., "strict": true, "schema": {...}}` |
| Chat Completions | `response_format` — `{"type": "json_schema", "json_schema": {"name": ..., "strict": true, "schema": {...}}}` |

**These routes are gated per model, and Claude models are not routable on them.** Each API
is served under two path prefixes, `/openai/v1/...` and `/v1/...`, and which prefix a model
accepts varies by model (in testing, OpenAI GPT models were served under `/openai/v1/` and
open-weight models under `/v1/`). A Claude ID returns
`400 The model '...' does not support the '/openai/v1/responses' API`, or the equivalent
for the other paths — a 400 rather than a 404, because the model exists on the endpoint
but not on that path. For Claude, use the APIs in Parts 1–3.

Two further things to know if you do use these routes with a non-Claude model:

- Model IDs follow the same short, unprefixed Mantle convention. A `global.` prefix 404s.
- **The schema validation is looser there.** Requests that the Claude grammar compiler
  rejects — omitting `additionalProperties`, numeric bounds like `minimum`, recursive
  `$ref` — returned `200 OK` in testing with an open-weight model. Do not carry
  conclusions about supported schema keywords from one surface to the other; the lists
  above are specific to Claude.

---

## Summary

| | `Converse` | `InvokeModel` | Mantle Messages |
|---|---|---|---|
| Schema location | `outputConfig.textFormat.structure.jsonSchema.schema` | `output_config.format.schema` | `output_config.format.schema` |
| Schema type | JSON **string** | **dict** | **dict** |
| `name` / `description` | accepted | rejected | rejected |
| Model ID form | `global.`/geo prefix | `global.`/geo prefix | short, unprefixed |
| Tool schema key | `toolSpec.inputSchema.json` (dict) | `input_schema` (dict) | `input_schema` (dict) |
| Strict tools flag | `toolSpec.strict` | `tools[].strict` | `tools[].strict` |
| Token budget | `inferenceConfig.maxTokens` | `max_tokens` in body | `max_tokens` kwarg |
| Refusal stop reason | not present | `refusal` | `refusal` |

Pick the surface by what else you need: `Converse` if you want one request shape across
model providers, `InvokeModel` or Mantle if you want Anthropic-native features and field
names. The structured-output guarantee is the same on all three.

### Cost

Every cell here makes a real Bedrock inference call billed at standard per-token rates for
the model in `RUNTIME_MODEL` / `MANTLE_MODEL`; the whole notebook is a handful of short
requests. There is no infrastructure to tear down. Structured outputs also inject a short
system prompt describing the expected format, so input token counts run slightly higher
than your prompt alone.

### Further reading

- [Structured outputs — Anthropic docs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)
- [Strict tool use — Anthropic docs](https://platform.claude.com/docs/en/agents-and-tools/tool-use/strict-tool-use)
- [Structured outputs — Amazon Bedrock user guide](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html)
- [Converse API reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html)
- [Bedrock inference profiles](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-profiles-support.html)